# Original Learned Metrics
Original workflow: `old/GPN_change.ipynb`. Imports now use the `amgpn` package.
Run from the repository root after an editable install. Configuration keys retain the original notebook/cell identifiers.


In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#GPN模型迭代：在原模型基础上改进

from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
from amgpn.legacy.original.episodes import *
from amgpn.legacy.original.learned_metrics import GPN,GPNTrainer,count_parameters

# def main():
#     # 参数设置
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
    
#     # 数据路径
#     # current_path=Path(__file__).resolve().parent
# Historical configuration omitted; provide values through private configuration.
    
#     # 创建原始数据集
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
    
#     # 创建元任务划分集
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
    
#     # DataLoader
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
#     #创建模型
# Historical configuration omitted; provide values through private configuration.

#     model_without_se = GPN(use_se=True)
#     params_without_se = count_parameters(model_without_se)
#     # 创建训练器
# Historical configuration omitted; provide values through private configuration.
#     print("开始训练...")
#     meta_learner.train(train_loader=train_loader,test_loader=test_loader,
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
    
#     print("训练完成！")

#     meta_learner.evaluate(test_loader=test_loader,n_ways=N_WAY)

# if __name__ == '__main__':
#     main()
# 参数设置
FEATURE_DIM = _cfg_require('old/GPN_change.ipynb.cell0.module.FEATURE_DIM')
N_WAY = _cfg_require('old/GPN_change.ipynb.cell0.module.N_WAY')
K_SHOT = _cfg_require('old/GPN_change.ipynb.cell0.module.K_SHOT')
Q_QUERY = _cfg_require('old/GPN_change.ipynb.cell0.module.Q_QUERY')
NUM_TASKS = _cfg_require('old/GPN_change.ipynb.cell0.module.NUM_TASKS')
EPOCHS = _cfg_require('old/GPN_change.ipynb.cell0.module.EPOCHS')
LR = _cfg_require('old/GPN_change.ipynb.cell0.module.LR')
BATCH_SIZE = _cfg_require('old/GPN_change.ipynb.cell0.module.BATCH_SIZE')
SEED = _cfg_require('old/GPN_change.ipynb.cell0.module.SEED')

# 数据路径
# current_path=Path(__file__).resolve().parent
dataset_path = _cfg_require('old/GPN_change.ipynb.cell0.module.dataset_path')

SAVE_MODEL_PATH=_cfg_require('old/GPN_change.ipynb.cell0.module.SAVE_MODEL_PATH')
def main():
    
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    train_dataset=UAVDataset(data_dir_path=dataset_path+_cfg_require('old/GPN_change.ipynb.cell0.main.path'),seed=SEED,max_sample_count=_cfg_require('old/GPN_change.ipynb.cell0.main.max_sample_count'))
    test_dataset=UAVDataset(data_dir_path=dataset_path+_cfg_require('old/GPN_change.ipynb.cell0.main.path__2'),seed=SEED,max_sample_count=_cfg_require('old/GPN_change.ipynb.cell0.main.max_sample_count__2'))

    
    # 创建元任务划分集
    meta_train_dataset = MetaDataset(train_dataset, 8,1,5,batch_size=BATCH_SIZE,num_tasks=NUM_TASKS,epochs=EPOCHS)
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,batch_size=BATCH_SIZE,num_tasks=_cfg_require('old/GPN_change.ipynb.cell0.main.num_tasks'),epochs=_cfg_require('old/GPN_change.ipynb.cell0.main.epochs'))

    
    # DataLoader
    train_loader = DataLoader(meta_train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=_cfg_require('old/GPN_change.ipynb.cell0.main.num_workers'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('old/GPN_change.ipynb.cell0.main.batch_size'), shuffle=False)

    #创建模型
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = GPN(use_se=_cfg_require('old/GPN_change.ipynb.cell0.main.use_se'))#GPN_change
    # model = GPN_Optimized(use_repvgg=True, use_convnext=True)#GPN_new
    # model = GPN_Optimized(use_repvgg=False,
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    params = count_parameters(model)
    
    
    x=torch.randn(2,1,_cfg_require('old/GPN_change.ipynb.cell0.main.size_or_budget'),_cfg_require('old/GPN_change.ipynb.cell0.main.size_or_budget__2'))
    # 前向传播
    v, s = model(x)
    print(f"Embedding features (v) shape: {v.shape}")
    print(f"Precision features (s) shape: {s.shape}")
    print(f"Precision values are positive: {(s > 0).all().item()}")

    # 带中间特征输出
    v, s, features = model(x, return_intermediate=True)
    print(f"\nIntermediate features:")
    for name, feat in features.items():
        if isinstance(feat, torch.Tensor):
            print(f"  {name}: {feat.shape}")

    # 创建训练器
    # meta_learner = GPNTrainer(model,device=device,loss_feature_dim=LOSS_FEATURE_DIM,lr=BLR,metric_type='mahalanobis')
    meta_learner = GPNTrainer(model,device=device,loss_feature_dim=FEATURE_DIM,lr=LR)
    print("开始训练...")
    meta_learner.train(train_loader=train_loader,test_loader=test_loader,num_tasks=NUM_TASKS,n_ways=N_WAY,epochs=EPOCHS,save_path=SAVE_MODEL_PATH)

    print("训练完成！")

    # meta_learner.evaluate(test_loader=test_loader,n_ways=N_WAY)
    
def inference():
    '\n    推理示例：加载已保存的模型进行预测\n    \n    !!!若报模型结构错误，检查：<configured>.引用是否为对应模型\n    '
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 创建模型
    test_dataset=UAVDataset(data_dir_path=_cfg_require('old/GPN_change.ipynb.cell0.inference.data_dir_path'),seed=SEED,max_sample_count=Q_QUERY+10)
    # Historical configuration omitted; provide values through private configuration.

    model = GPN(use_se=_cfg_require('old/GPN_change.ipynb.cell0.inference.use_se'))
    # model = GPN_Optimized(use_repvgg=False,
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # 创建trainer并加载模型
    trainer = GPNTrainer(model, device=device)
    # trainer = GPNTrainer(model,device,loss_feature_dim=LOSS_FEATURE_DIM, lr=BLR,metric_type='mahalanobis')
    trainer.load_model(_cfg_require('old/GPN_change.ipynb.cell0.inference.path'))
    
    # 评估
    results = trainer.full_evaluation(test_dataset, n_trials=_cfg_require('old/GPN_change.ipynb.cell0.inference.n_trials'),q_query=Q_QUERY)
    plot_all_visualizations(results,output_dir=_cfg_require('old/GPN_change.ipynb.cell0.inference.output_dir'))#V2_rnx+rnx_d_ft5:1_L_8_1_5
    print("推理完成")
if __name__ == '__main__':
    main()
    inference()
